In [1]:
import pandas as pd 
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [3]:
df = pd.read_csv("customer_shopping_behavior.csv")

In [4]:
category_median = df.groupby('Category')['Review Rating'].transform('median')
df['Review Rating'] = df['Review Rating'].fillna(category_median)

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [6]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

In [7]:
frequency_mapping = {
    'Fortnightly': 14, 'Weekly': 7, 'Monthly': 30, 'Quarterly': 90,
    'Bi-Weekly': 14, 'Annually': 365, 'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# 6. Drop redundant column
df = df.drop('promo_code_used', axis=1)

print("Data cleaned successfully! Preparing to load into database...")

Data cleaned successfully! Preparing to load into database...


In [8]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365


In [ ]:
# 1. Connection Details
username = "postgres"
password = "your password here"  # Put your PostgreSQL password here
host = "localhost"
port = "5432"
database = "customer_behavior"   # The database you created in DBeaver

# 2. Create the connection URL safely (prevents @ symbol errors)
connection_url = URL.create(
    "postgresql+psycopg2",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database,
)
engine = create_engine(connection_url)

# 3. Push the cleaned DataFrame into the SQL database
table_name = "customer" 
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Success! Data loaded into table '{table_name}' in the '{database}' database.")

Success! Data loaded into table 'customer' in the 'customer_behavior' database.
